[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brilliantbeaver/alexpose/blob/main/penny/gavd3/08_gait_parameter_probing.ipynb)

# 08. Probe the frozen encoder for gait biomechanics

Compute gait proxies from the raw cached poses, then ask whether the frozen S-JEPA latents make them linearly decodable. RidgeCV probes report R2 against two baselines, and a per-frame probe hunts for a latent phase clock.

**Research use only.** This tutorial does not diagnose a person or validate a clinical device.

**Run it:** locally, use `uv sync` then `uv run jupyter lab` from this folder. In Colab, use the badge and run the setup cell. Restart the kernel after changing `penny/gavd3/.env`.

**Keep the walk visible:** notebook 01 opens the source video, and notebook 02 shows frame, bbox, and skeleton alignment. Notebook 05 introduced the pooled latents and notebook 06 fitted classifiers on them. This notebook reuses the same pooling contract for linear regression probes, so check notebook 05 whenever a pooling detail looks surprising.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/brilliantbeaver/alexpose.git"

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
        "seaborn", "torch", "tqdm", "python-dotenv", "yt-dlp[default]",
        "opencv-python-headless", "mediapipe<1", "joblib", "pyarrow",
    ])
    clone_dir = Path("/content/alexpose")
    if not (clone_dir / ".git").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])
    os.chdir(clone_dir)


def find_project_root(start=None):
    env_root = os.getenv("ALEXPOSE_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
        print(f"Ignoring invalid ALEXPOSE_ROOT: {candidate}")
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / "penny" / "gavd3"

try:
    from dotenv import load_dotenv
    load_dotenv(TUTORIAL_DIR / ".env", override=False)
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except Exception:
    pass

MODE = os.getenv("GAVD3_MODE", "smoke").strip().lower()
if MODE not in {"smoke", "real"}:
    raise ValueError("GAVD3_MODE must be smoke or real")
if MODE == "smoke":
    print(
        "SMOKE MODE: hand-authored motions test code paths only. "
        "They have no pathophysiological or clinical validity."
    )

PREFERRED_ROOT = Path(
    os.getenv(
        "GAVD4_ROOT",
        "/Users/pmui/vaults/worldmodels/gait/skeleton-jepa/gavd4",
    )
).expanduser()

requested_data = os.getenv("GAVD4_DATA_DIR") or os.getenv("GAVD_DATA_GAVD_DIR")
if requested_data and Path(requested_data).expanduser().exists():
    DATA_GAVD_DIR = Path(requested_data).expanduser()
elif requested_data:
    print(f"Ignoring missing GAVD CSV path: {Path(requested_data).expanduser()}")
    if (PREFERRED_ROOT / "data-gavd").exists():
        DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
    else:
        DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"
elif (PREFERRED_ROOT / "data-gavd").exists():
    DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
else:
    DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"

requested_youtube = os.getenv("GAVD4_YOUTUBE_DIR") or os.getenv("GAVD_YOUTUBE_DIR")
if requested_youtube:
    YOUTUBE_DIR = Path(requested_youtube).expanduser()
elif PREFERRED_ROOT.exists():
    YOUTUBE_DIR = PREFERRED_ROOT / "youtube"
else:
    YOUTUBE_DIR = PROJECT_ROOT / "penny" / "gavd3" / "work" / "youtube"

CACHE_DIR = Path(
    os.getenv("GAVD3_CACHE_DIR", TUTORIAL_DIR / "work" / "cache")
).expanduser()
ARTIFACT_ROOT = Path(
    os.getenv("GAVD3_ARTIFACT_DIR", TUTORIAL_DIR / "work" / "artifacts")
).expanduser()
ARTIFACT_DIR = ARTIFACT_ROOT / MODE
POSE_DIR = ARTIFACT_DIR / "poses"

for folder in [CACHE_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR / "xdg-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import SVG, display


def show_tutorial_svg(filename):
    '''Render a repository SVG reliably in local Jupyter and Colab.'''
    path = TUTORIAL_DIR / "images" / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing tutorial figure {path}. Clone the full alexpose repository."
        )
    display(SVG(filename=str(path)))

print(f"mode: {MODE}")
print(f"project: {PROJECT_ROOT}")
print(f"GAVD CSVs: {DATA_GAVD_DIR}")
print(f"YouTube cache: {YOUTUBE_DIR}")
print(f"artifacts: {ARTIFACT_DIR}")

## Why probe the frozen encoder

Self-supervised pretraining never sees a label. S-JEPA learns to predict hidden joint-time patches in latent space from the visible ones. The question this notebook carries into the BrainBodyFM workshop is the measurement one: does that self-supervised objective leave a representation a downstream user can actually read? In other words, does pretraining help decode gait biomechanics, or does it merely rearrange coordinates?

A linear probe is the smallest possible downstream reader. You freeze the target encoder, pool one vector per sequence, and fit a regularized linear map from that vector to a quantity such as cadence. The out-of-fold R2 tells you how much of that quantity is accessible through a single linear readout. High R2 means pretraining organized the latent so that the property sits along a straight line in embedding space. Low R2 means the property is entangled or absent.

R2 alone is not evidence. A probe can succeed for the wrong reason, so every probe is compared with two baselines fitted by the identical procedure:

1. pooled statistics of the raw prepared coordinates, a coordinate-level baseline that contains no learned representation, and
2. pose-missingness features, which capture where the pose detector lost joints or frames.

If the frozen latents beat both baselines, the gain is gait signal rather than tracking artifacts. This notebook computes the gait targets from the RAW cached poses, then runs exactly this comparison. All numbers are research quality only: the corpus is small, several sequences share a source video, and every gait quantity below is a video-based proxy, never a clinical measurement.

In [ ]:
BLAZEPOSE_33 = [
    "NOSE", "LEFT_EYE_INNER", "LEFT_EYE", "LEFT_EYE_OUTER",
    "RIGHT_EYE_INNER", "RIGHT_EYE", "RIGHT_EYE_OUTER", "LEFT_EAR",
    "RIGHT_EAR", "MOUTH_LEFT", "MOUTH_RIGHT", "LEFT_SHOULDER",
    "RIGHT_SHOULDER", "LEFT_ELBOW", "RIGHT_ELBOW", "LEFT_WRIST",
    "RIGHT_WRIST", "LEFT_PINKY", "RIGHT_PINKY", "LEFT_INDEX",
    "RIGHT_INDEX", "LEFT_THUMB", "RIGHT_THUMB", "LEFT_HIP",
    "RIGHT_HIP", "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE",
    "RIGHT_ANKLE", "LEFT_HEEL", "RIGHT_HEEL", "LEFT_FOOT_INDEX",
    "RIGHT_FOOT_INDEX",
]
MASK_KEYPOINTS = [11, 12, 23, 24, 25, 26, 27, 28, 31, 32]
assert [BLAZEPOSE_33[i] for i in MASK_KEYPOINTS] == [
    "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HIP", "RIGHT_HIP",
    "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE", "RIGHT_ANKLE",
    "LEFT_FOOT_INDEX", "RIGHT_FOOT_INDEX",
]



CONDITIONS = ["normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"]

In [ ]:
def synthetic_gait_sequence(condition="normal", frames=64, seed=0):
    '''Create a code-path fixture, not a physiological disease simulation.'''
    rng = np.random.default_rng(seed)
    phase = np.linspace(0.0, 4.0 * np.pi, frames, endpoint=False)
    seq = np.zeros((frames, 33, 4), dtype=np.float32)
    seq[..., 3] = 1.0
    base = {
        11: (0.42, 0.28), 12: (0.58, 0.28),
        23: (0.45, 0.52), 24: (0.55, 0.52),
        25: (0.44, 0.70), 26: (0.56, 0.70),
        27: (0.43, 0.89), 28: (0.57, 0.89),
        29: (0.42, 0.92), 30: (0.58, 0.92),
        31: (0.39, 0.94), 32: (0.61, 0.94),
    }
    for joint, (x, y) in base.items():
        seq[:, joint, 0] = x
        seq[:, joint, 1] = y
    amplitude = 0.045
    lift = 0.025
    if condition == "parkinsons":
        amplitude *= 0.45
        lift *= 0.45
    if condition == "myopathic":
        seq[:, [11, 12], 0] += 0.03 * np.sin(phase)[:, None]
        seq[:, [23, 24], 0] += 0.018 * np.sin(phase)[:, None]
    for joint, knee, foot, offset in [(27, 25, 31, 0.0), (28, 26, 32, np.pi)]:
        wave = np.sin(phase + offset)
        if condition == "stroke" and joint == 27:
            wave = 0.35 * wave
        if condition == "cerebralpalsy":
            seq[:, knee, 1] -= 0.045
            seq[:, joint, 1] -= 0.02
        seq[:, joint, 0] += amplitude * wave
        seq[:, knee, 0] += 0.4 * amplitude * wave
        seq[:, foot, 0] += amplitude * wave
        seq[:, joint, 1] -= lift * np.maximum(wave, 0.0)
        seq[:, foot, 1] -= 0.7 * lift * np.maximum(wave, 0.0)
    seq[..., :3] += rng.normal(0.0, 0.0025, seq[..., :3].shape)
    return seq


def synthetic_corpus(conditions=None, n_per_condition=10, frames=64, seed=42):
    if conditions is None:
        conditions = [
            "normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"
        ]
    records = []
    counter = 0
    for condition in conditions:
        for sample in range(n_per_condition):
            records.append({
                "condition": condition,
                "sequence_id": f"smoke_{condition}_{sample:03d}",
                "video_id": f"smoke_video_{condition}_{sample // 2:02d}",
                "sequence": synthetic_gait_sequence(
                    condition=condition,
                    frames=frames,
                    seed=seed + counter,
                ),
            })
            counter += 1
    return records

In [ ]:
def interpolate_low_visibility(sequence, threshold=0.45, max_gap=4):
    '''Fill only short internal gaps and preserve the original validity mask.

    Long gaps and sequence ends are never extrapolated. Their coordinates remain
    missing until center_and_scale converts them to an explicit zero sentinel.
    They can never become S-JEPA prediction targets.
    '''
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
        raise ValueError(f"Expected [T, 33, 4], received {sequence.shape}")
    visibility = np.nan_to_num(sequence[..., 3], nan=0.0)
    finite = np.isfinite(sequence[..., :3]).all(axis=-1)
    valid = (visibility >= threshold) & finite
    filled = valid.copy()
    for joint in range(33):
        observed = np.flatnonzero(valid[:, joint])
        for left, right in zip(observed[:-1], observed[1:]):
            gap = int(right - left - 1)
            if not 0 < gap <= max_gap:
                continue
            fraction = (
                np.arange(1, gap + 1, dtype=np.float32) / (gap + 1)
            )[:, None]
            sequence[left + 1:right, joint, :3] = (
                sequence[left, joint, :3][None, :] * (1.0 - fraction)
                + sequence[right, joint, :3][None, :] * fraction
            )
            filled[left + 1:right, joint] = True
        sequence[~filled[:, joint], joint, :3] = np.nan
    sequence[..., 3] = visibility
    return sequence, valid


def center_and_scale(sequence, eps=1e-6):
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    xyz = sequence[..., :3]
    left_hip, right_hip = xyz[:, 23], xyz[:, 24]
    left_ok = np.isfinite(left_hip).all(axis=1)
    right_ok = np.isfinite(right_hip).all(axis=1)
    pelvis = np.full((len(xyz), 3), np.nan, dtype=np.float32)
    pelvis[left_ok & right_ok] = 0.5 * (
        left_hip[left_ok & right_ok] + right_hip[left_ok & right_ok]
    )
    pelvis[left_ok & ~right_ok] = left_hip[left_ok & ~right_ok]
    pelvis[right_ok & ~left_ok] = right_hip[right_ok & ~left_ok]
    pelvis_ok = np.isfinite(pelvis).all(axis=1)
    fallback = np.median(pelvis[pelvis_ok], axis=0) if pelvis_ok.any() else np.zeros(3)
    pelvis[~np.isfinite(pelvis).all(axis=1)] = fallback
    xyz = xyz - pelvis[:, None, :]
    shoulder_width = np.linalg.norm(xyz[:, 11, :2] - xyz[:, 12, :2], axis=-1)
    hip_width = np.linalg.norm(xyz[:, 23, :2] - xyz[:, 24, :2], axis=-1)
    body_scale = np.nanmedian(np.maximum(shoulder_width, hip_width))
    if not np.isfinite(body_scale) or body_scale < eps:
        body_scale = 1.0
    sequence[..., :3] = np.nan_to_num(
        xyz / body_scale, nan=0.0, posinf=0.0, neginf=0.0
    )
    return np.nan_to_num(sequence, nan=0.0, posinf=0.0, neginf=0.0)


def temporal_resize(array, frames):
    array = np.asarray(array)
    if len(array) == frames:
        return array.copy()
    if len(array) < 2:
        return np.repeat(array, frames, axis=0)
    old_t = np.linspace(0.0, 1.0, len(array))
    new_t = np.linspace(0.0, 1.0, frames)
    flat = array.reshape(len(array), -1)
    resized = np.stack(
        [np.interp(new_t, old_t, flat[:, i]) for i in range(flat.shape[1])],
        axis=1,
    )
    return resized.reshape(frames, *array.shape[1:]).astype(array.dtype)


def prepare_sequence(
    sequence,
    frames=64,
    visibility_threshold=0.45,
    max_gap=4,
):
    cleaned, valid = interpolate_low_visibility(
        sequence, visibility_threshold, max_gap=max_gap
    )
    cleaned = center_and_scale(cleaned)
    cleaned = temporal_resize(cleaned, frames)
    valid = temporal_resize(valid.astype(np.float32), frames) >= 0.5
    return cleaned[..., :3].astype(np.float32), valid.astype(bool)

In [ ]:
def uniform_neurologic_mask(valid_patch, mask_fraction=0.60, seed=None):
    """Sample eligible joint-time tokens uniformly, without motion scores.

    valid_patch has shape [B, S, V]. True means that a patch can be a target.
    The returned mask has the same shape. True means hidden from the view encoder.
    """
    valid_patch = np.asarray(valid_patch, dtype=bool)
    if valid_patch.ndim != 3 or valid_patch.shape[2] != 33:
        raise ValueError(f"Expected [B, S, 33], received {valid_patch.shape}")
    if not 0.0 < mask_fraction < 1.0:
        raise ValueError("mask_fraction must be between 0 and 1")
    rng = np.random.default_rng(seed)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    counts = eligible.reshape(len(eligible), -1).sum(axis=1)
    if np.any(counts < 2):
        raise ValueError("Each sample needs at least two valid eligible tokens")
    n_mask = max(1, int(np.floor(counts.min() * mask_fraction)))
    n_mask = min(n_mask, int(counts.min()) - 1)
    mask = np.zeros_like(eligible)
    for batch_index in range(len(mask)):
        candidates = np.flatnonzero(eligible[batch_index].reshape(-1))
        chosen = rng.choice(candidates, size=n_mask, replace=False)
        mask[batch_index].reshape(-1)[chosen] = True
    forbidden = sorted(set(range(33)).difference(MASK_KEYPOINTS))
    assert not mask[:, :, forbidden].any()
    assert mask.reshape(len(mask), -1).any(axis=1).all()
    assert (~mask).reshape(len(mask), -1).any(axis=1).all()
    return mask


def mask_audit(mask, valid_patch):
    mask = np.asarray(mask, dtype=bool)
    valid_patch = np.asarray(valid_patch, dtype=bool)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    masked_counts = mask.reshape(len(mask), -1).sum(axis=1)
    eligible_counts = eligible.reshape(len(mask), -1).sum(axis=1)
    per_sample_ratio = masked_counts / eligible_counts
    touched = np.flatnonzero(mask.any(axis=(0, 1))).tolist()
    return {
        "masked_keypoints": touched,
        "masked_names": [BLAZEPOSE_33[i] for i in touched],
        "global_fraction": float(mask.mean()),
        "eligible_mask_fraction_min": float(per_sample_ratio.min()),
        "eligible_mask_fraction_mean": float(per_sample_ratio.mean()),
        "eligible_mask_fraction_max": float(per_sample_ratio.max()),
        "forbidden_count": int(mask[:, :, sorted(set(range(33)) - set(MASK_KEYPOINTS))].sum()),
    }

In [ ]:
import copy
import math
import torch
from torch import nn


class SkeletonPatchEncoder(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        if frames % segment_length:
            raise ValueError("frames must be divisible by segment_length")
        self.frames = frames
        self.joints = joints
        self.coordinate_dim = coordinate_dim
        self.segment_length = segment_length
        self.segments = frames // segment_length
        self.embed_dim = embed_dim
        self.patch_embed = nn.Linear(segment_length * coordinate_dim, embed_dim)
        self.time_pos = nn.Parameter(torch.randn(self.segments, embed_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

    def patchify(self, x):
        batch, frames, joints, channels = x.shape
        expected = (self.frames, self.joints, self.coordinate_dim)
        if (frames, joints, channels) != expected:
            raise ValueError(f"Expected [B, {expected}], received {x.shape}")
        patches = x.reshape(
            batch, self.segments, self.segment_length, joints, channels
        )
        patches = patches.permute(0, 1, 3, 2, 4).contiguous()
        return patches.flatten(3)

    def positioned_tokens(self, x):
        tokens = self.patch_embed(self.patchify(x))
        return (
            tokens
            + self.time_pos[None, :, None, :]
            + self.joint_pos[None, None, :, :]
        )

    def forward(self, x, keep_mask=None):
        tokens = self.positioned_tokens(x)
        batch = len(tokens)
        flat = tokens.reshape(batch, self.segments * self.joints, self.embed_dim)
        if keep_mask is not None:
            keep_mask = keep_mask.reshape(batch, -1)
            kept_per_sample = keep_mask.sum(dim=1)
            if not torch.equal(kept_per_sample, kept_per_sample[:1].expand_as(kept_per_sample)):
                raise ValueError("Each sample must keep the same number of tokens")
            flat = flat[keep_mask].reshape(batch, int(kept_per_sample[0]), self.embed_dim)
        return self.norm(self.blocks(flat))


class SkeletonPredictor(nn.Module):
    def __init__(
        self,
        segments,
        joints,
        encoder_dim=64,
        predictor_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        self.segments = segments
        self.joints = joints
        self.encoder_to_predictor = nn.Linear(encoder_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.normal_(self.mask_token, std=0.02)
        self.time_pos = nn.Parameter(torch.randn(segments, predictor_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, predictor_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=predictor_dim,
            nhead=heads,
            dim_feedforward=predictor_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(predictor_dim)
        self.output = nn.Linear(predictor_dim, encoder_dim)

    def forward(self, visible_features, target_mask):
        batch = len(visible_features)
        target_mask = target_mask.reshape(batch, self.segments * self.joints)
        visible_mask = ~target_mask
        visible = self.encoder_to_predictor(visible_features)
        full = self.mask_token.expand(
            batch, self.segments * self.joints, -1
        ).clone()
        full[visible_mask] = visible.reshape(-1, visible.shape[-1])
        positions = (
            self.time_pos[:, None, :] + self.joint_pos[None, :, :]
        ).reshape(1, self.segments * self.joints, -1)
        full = full + positions
        predicted = self.output(self.norm(self.blocks(full)))
        return predicted[target_mask].reshape(batch, -1, predicted.shape[-1])


class SJEPAGait(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        encoder_depth=2,
        predictor_depth=2,
        heads=4,
    ):
        super().__init__()
        self.view_encoder = SkeletonPatchEncoder(
            frames, joints, coordinate_dim, segment_length,
            embed_dim, encoder_depth, heads,
        )
        self.target_encoder = copy.deepcopy(self.view_encoder)
        for parameter in self.target_encoder.parameters():
            parameter.requires_grad_(False)
        self.predictor = SkeletonPredictor(
            self.view_encoder.segments,
            joints,
            embed_dim,
            embed_dim,
            predictor_depth,
            heads,
        )
        self.register_buffer("target_center", torch.zeros(embed_dim))

    def forward(self, view, target, target_mask):
        visible_features = self.view_encoder(view, keep_mask=~target_mask)
        predicted = self.predictor(visible_features, target_mask)
        with torch.no_grad():
            all_targets = self.target_encoder(target)
            flat_mask = target_mask.reshape(len(target), -1)
            selected = all_targets[flat_mask].reshape(
                len(target), -1, all_targets.shape[-1]
            )
        return predicted, selected

    @torch.no_grad()
    def update_target(self, momentum):
        for target_parameter, view_parameter in zip(
            self.target_encoder.parameters(), self.view_encoder.parameters()
        ):
            target_parameter.mul_(momentum).add_(
                view_parameter, alpha=1.0 - momentum
            )

    @torch.no_grad()
    def update_center(self, targets, beta=0.9):
        batch_center = targets.mean(dim=(0, 1))
        self.target_center.mul_(beta).add_(batch_center, alpha=1.0 - beta)


def sjepa_cross_entropy(
    predicted,
    targets,
    center,
    predictor_temperature=0.10,
    target_temperature=0.06,
):
    target_prob = torch.softmax(
        (targets - center[None, None, :]) / target_temperature,
        dim=-1,
    ).detach()
    prediction_log_prob = torch.log_softmax(
        predicted / predictor_temperature,
        dim=-1,
    )
    return -(target_prob * prediction_log_prob).sum(dim=-1).mean()


def cosine_ema(step, total_steps, start=0.996, end=1.0):
    progress = min(max(step / max(total_steps - 1, 1), 0.0), 1.0)
    return end - (end - start) * (math.cos(math.pi * progress) + 1.0) / 2.0


LEFT_RIGHT_PAIRS = [
    (1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12),
    (13, 14), (15, 16), (17, 18), (19, 20), (21, 22),
    (23, 24), (25, 26), (27, 28), (29, 30), (31, 32),
]


def geometric_view(
    x,
    max_degrees=8.0,
    translate=0.03,
    flip_probability=0.0,
):
    """Apply one sequence-wide transform per sample.

    Rotation is around the relative vertical y axis, so x and z are mixed.
    Flip defaults to off because laterality can matter for stroke. If enabled,
    coordinates are reflected and every left-right landmark pair is swapped.
    """
    view = x.clone()
    present = view.abs().sum(dim=-1) > 1e-8
    batch = len(view)
    angles = (
        torch.rand(batch, device=x.device) * 2.0 - 1.0
    ) * math.radians(max_degrees)
    cosine, sine = torch.cos(angles), torch.sin(angles)
    original_x = view[..., 0].clone()
    original_z = view[..., 2].clone()
    rotated_x = cosine[:, None, None] * original_x + sine[:, None, None] * original_z
    rotated_z = -sine[:, None, None] * original_x + cosine[:, None, None] * original_z
    view[..., 0] = rotated_x
    view[..., 2] = rotated_z
    offsets = (torch.rand(batch, 1, 1, 2, device=x.device) * 2.0 - 1.0) * translate
    view[..., :2] += offsets
    if flip_probability > 0:
        flip = torch.rand(batch, device=x.device) < flip_probability
        for batch_index in torch.where(flip)[0].tolist():
            view[batch_index, ..., 0] *= -1.0
            original = view[batch_index].clone()
            original_present = present[batch_index].clone()
            for left, right in LEFT_RIGHT_PAIRS:
                view[batch_index, :, left] = original[:, right]
                view[batch_index, :, right] = original[:, left]
                present[batch_index, :, left] = original_present[:, right]
                present[batch_index, :, right] = original_present[:, left]
    view = view.masked_fill(~present[..., None], 0.0)
    return view

In [ ]:
def pose_records_from_cache(pose_dir=POSE_DIR, conditions=CONDITIONS):
    records = []
    for condition in conditions:
        folder = Path(pose_dir) / condition
        for path in sorted(folder.glob("*.npz")):
            data = np.load(path, allow_pickle=False)
            required = {
                "sequence", "sequence_id", "video_id", "condition",
                "frame_numbers", "crop_bounds", "fps", "source_csv",
                "source_video", "pose_model", "pose_model_sha256",
                "extraction_version",
            }
            missing = required.difference(data.files)
            if missing:
                raise ValueError(
                    f"Stale pose cache {path} is missing {sorted(missing)}. "
                    "Re-extract it with notebook 02."
                )
            sequence = data["sequence"].astype(np.float32)
            if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
                raise ValueError(f"Bad pose shape in {path}: {sequence.shape}")
            stored_condition = str(data["condition"].item())
            if stored_condition != condition:
                raise ValueError(
                    f"Pose condition {stored_condition} does not match folder {condition}"
                )
            if len(data["frame_numbers"]) != len(sequence):
                raise ValueError(f"Frame and pose lengths differ in {path}")
            records.append({
                "condition": condition,
                "sequence_id": str(data["sequence_id"].item()),
                "video_id": str(data["video_id"].item()),
                "source_video": str(data["source_video"].item()),
                "fps": float(data["fps"].item()),
                "extraction_version": str(data["extraction_version"].item()),
                "pose_model_sha256": str(data["pose_model_sha256"].item()),
                "sequence": sequence,
                "path": str(path),
            })
    return records


def load_records_for_mode(conditions=CONDITIONS, smoke_per_condition=10, frames=64):
    if MODE == "smoke":
        records = synthetic_corpus(
            conditions=conditions,
            n_per_condition=smoke_per_condition,
            frames=frames,
        )
        print(f"Explicit smoke corpus: {len(records)} synthetic sequences")
        return records
    records = pose_records_from_cache(conditions=conditions)
    counts = pd.Series([r["condition"] for r in records]).value_counts()
    missing = [condition for condition in conditions if counts.get(condition, 0) == 0]
    if missing:
        raise FileNotFoundError(
            f"Real mode requires cached pose sequences for {missing}. "
            "Run notebook 02 first."
        )
    print(f"Real pose corpus: {len(records)} sequences")
    return records

## Load the frozen checkpoint

This notebook never constructs an untrained encoder as a fallback. Run notebook 04 first. Checkpoints are stored per mode, and the mode plus the masked keypoint set are verified again here so that a smoke checkpoint cannot be mixed with real poses, or the reverse.

In [ ]:
checkpoint_path = ARTIFACT_DIR / "sjepa_normal.pt"
if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Missing {checkpoint_path}. Run notebook 04 in {MODE} mode."
    )
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)
if checkpoint["mode"] != MODE:
    raise ValueError(
        f"Checkpoint mode {checkpoint['mode']} does not match {MODE}"
    )
if checkpoint["mask_keypoints"] != MASK_KEYPOINTS:
    raise ValueError("Checkpoint mask set does not match this tutorial")
model = SJEPAGait(**checkpoint["config"])
model.load_state_dict(checkpoint["model_state"])
model.eval()
config = checkpoint["config"]
FRAMES = config["frames"]
SEGMENT_LENGTH = config["segment_length"]
LATENT_DIM = config["embed_dim"]
print("checkpoint fingerprint:", checkpoint["dataset_fingerprint"])
print(
    f"encoder: {LATENT_DIM}-d tokens, {FRAMES // SEGMENT_LENGTH} segments "
    f"of {SEGMENT_LENGTH} frames"
)

## Load every sequence and audit coverage

Every latent in this notebook comes from the complete unmasked EMA target encoder. The gait proxies keep the RAW, un-resized poses with their true frame count and fps. The same raw sequences are then prepared (cleaned, centered, resized to FRAMES) exactly as notebooks 05 and 06 did, so the pooled latents here follow the same pooling contract.

In [ ]:
records = load_records_for_mode(
    conditions=CONDITIONS,
    smoke_per_condition=10,
    frames=FRAMES,
)
prepared = [
    prepare_sequence(record["sequence"], frames=FRAMES)
    for record in records
]
all_xyz = np.stack([item[0] for item in prepared]).astype(np.float32)
all_valid = np.stack([item[1] for item in prepared])
labels = np.asarray([record["condition"] for record in records])
sequence_ids = np.asarray([record["sequence_id"] for record in records])
video_ids = np.asarray([record["video_id"] for record in records])
min_coverage = float(os.getenv("GAVD_MIN_NEURO_COVERAGE", "0.50"))
coverage_report = pd.DataFrame({
    "condition": labels,
    "sequence_id": sequence_ids,
    "neurologic_observed_fraction": all_valid[:, :, MASK_KEYPOINTS].mean(axis=(1, 2)),
})
display(
    coverage_report.groupby("condition")["neurologic_observed_fraction"]
    .agg(["min", "mean"])
)
if (coverage_report["neurologic_observed_fraction"] < min_coverage).any():
    raise ValueError("At least one sequence fails the neurologic coverage threshold")
print(f"prepared pose tensor: {all_xyz.shape}, validity tensor: {all_valid.shape}")

## Gait parameters from the raw cached poses

Every gait target below is computed from record["sequence"] BEFORE prepare_sequence, because cadence and phase need real timing (fps and the raw frame count) and because resizing would distort the ankle clock. The pose cache stores normalized image coordinates, so all quantities are unitless video proxies. The table states which raw signal each proxy uses.

| Parameter | Raw signal used | Proxy definition |
| --- | --- | --- |
| cadence_cycles_per_second | left ankle x around the pelvis | unwrapped Hilbert phase advance of the ankle clock, cycles per second |
| step_length_asymmetry | horizontal excursion range of each ankle | absolute left-right difference divided by the mean range |
| knee_excursion_radians | hip-knee-ankle 2-D angle | 95th minus 5th percentile of the angle over the clip |
| trunk_sway_normalized_units | pelvis x, detrended | standard deviation of lateral pelvis position |
| speed_proxy_normalized_per_second | pelvis translation per frame | mean per-frame pelvis displacement times fps |
| stance_swing_ratio_proxy | toe height above its own median | frames on the ground divided by frames clearly lifted |
| continuous phase | centered left-ankle x, Hilbert transform | per-frame angle mapped to [0, 2*pi) |

One branching decision needs to be explicit. Real cached sequences carry an fps and frame_numbers, so the code below uses the true fps, duration = frames / fps, and a Hilbert-derived phase. Smoke fixtures carry neither fps nor frame_numbers, so the smoke branch substitutes fps = 30.0, duration = len(sequence) / 30.0, and synthesizes the phase signal from the fixture generator rule (the fixtures are drawn from a linear clock of 0 to 4*pi radians over the clip). Smoke fixtures are code-path checks, not clinical gait data, and the branch is labelled again in the code.

In [ ]:
import scipy.signal


# --- Raw-pose gait proxy helpers ------------------------------------------
# Everything here works on RAW cached poses, that is on record["sequence"]
# BEFORE prepare_sequence normalizes and resizes them. Short detector gaps are
# bridged by linear interpolation so the timing series stay dense. This is a
# video-based proxy pipeline, not a clinical measurement.

GAIT_PARAMETER_COLUMNS = [
    "cadence_cycles_per_second",
    "step_length_asymmetry",
    "knee_excursion_radians",
    "trunk_sway_normalized_units",
    "speed_proxy_normalized_per_second",
    "stance_swing_ratio_proxy",
]


def _interp_signal(values, good, length, max_gap=4):
    """Interpolate one coordinate across SHORT observed gaps only.

    Mirrors the prepare_sequence policy: gaps longer than max_gap stay NaN so
    the parameter computation never fabricates motion across long detector
    dropouts, and ends are clamped, never extrapolated.
    """
    index = np.flatnonzero(good)
    if len(index) < 2:
        return None
    signal = np.full(length, np.nan, dtype=np.float64)
    signal[index] = np.asarray(values, dtype=np.float64)[good]
    for left, right in zip(index[:-1], index[1:]):
        gap = int(right - left - 1)
        if not 0 < gap <= max_gap:
            continue
        fraction = np.arange(1, gap + 1, dtype=np.float64) / (gap + 1)
        signal[left + 1:right] = (
            signal[left] * (1.0 - fraction)
            + signal[right] * fraction
        )
    return signal


def _joint_xy(cleaned, valid, joint):
    """Interpolated x and y series for one joint, or (None, None)."""
    good = valid[:, joint].copy()
    x = cleaned[:, joint, 0]
    y = cleaned[:, joint, 1]
    good &= np.isfinite(x) & np.isfinite(y)
    return (
        _interp_signal(x, good, len(cleaned)),
        _interp_signal(y, good, len(cleaned)),
    )


def _detrend(signal):
    """Remove the linear walking translation with a least squares fit over
    the observed (finite) frames only; NaN gaps pass through untouched."""
    signal = np.asarray(signal, dtype=np.float64)
    finite = np.isfinite(signal)
    if int(finite.sum()) < 2:
        return signal
    time = np.arange(len(signal), dtype=np.float64)
    design = np.stack([time[finite], np.ones(int(finite.sum()))], axis=1)
    slope, intercept = np.linalg.lstsq(
        design, signal[finite], rcond=None
    )[0]
    return signal - (slope * time + intercept)
    coefficients, _, _, _ = np.linalg.lstsq(design, signal, rcond=None)
    return np.asarray(signal, dtype=np.float64) - design @ coefficients


def _smooth(signal, window=None):
    """Savitzky-Golay filter that degrades gracefully on short clips and
    never bridges NaN gaps: each contiguous finite run is smoothed on its
    own, and NaN frames pass through untouched."""
    signal = np.asarray(signal, dtype=np.float64)
    smoothed = np.full_like(signal, np.nan)
    finite = np.isfinite(signal)
    if not finite.any():
        return smoothed
    boundaries = np.flatnonzero(
        np.diff(np.concatenate([[0], finite.astype(int), [0]]))
    )
    for run_start, run_end in zip(boundaries[0::2], boundaries[1::2]):
        run = signal[run_start:run_end]
        length = len(run)
        win = min(9, length) if window is None else window
        if win % 2 == 0:
            win -= 1
        if win < 5 or win >= length:
            smoothed[run_start:run_end] = run
        else:
            smoothed[run_start:run_end] = scipy.signal.savgol_filter(
                run, win, polyorder=2
            )
    return smoothed


def _range95(signal):
    return float(np.nanpercentile(signal, 95) - np.nanpercentile(signal, 5))


def _pelvis_xy(cleaned, valid):
    """Midpoint of the two hips in raw normalized coordinates."""
    left_x, left_y = _joint_xy(cleaned, valid, 23)
    right_x, right_y = _joint_xy(cleaned, valid, 24)
    xs = [item for item in (left_x, right_x) if item is not None]
    ys = [item for item in (left_y, right_y) if item is not None]
    if not xs:
        return None, None
    return np.mean(np.stack(xs), axis=0), np.mean(np.stack(ys), axis=0)


def _relative_ankle_x(cleaned, valid, ankle, pelvis_x):
    """Ankle x relative to the pelvis, detrended (smoothed by the caller)."""
    ankle_x, _ = _joint_xy(cleaned, valid, ankle)
    if ankle_x is None or pelvis_x is None:
        return None
    return _detrend(ankle_x - pelvis_x)


def _knee_angle_series(cleaned, valid, hip, knee, ankle):
    """2-D knee angle proxy (hip-knee-ankle) in radians."""
    hip_x, hip_y = _joint_xy(cleaned, valid, hip)
    knee_x, knee_y = _joint_xy(cleaned, valid, knee)
    ankle_x, ankle_y = _joint_xy(cleaned, valid, ankle)
    if hip_x is None or knee_x is None or ankle_x is None:
        return None
    thigh_x = hip_x - knee_x
    thigh_y = hip_y - knee_y
    shank_x = ankle_x - knee_x
    shank_y = ankle_y - knee_y
    with np.errstate(divide="ignore", invalid="ignore"):
        cosine = (thigh_x * shank_x + thigh_y * shank_y) / (
            np.hypot(thigh_x, thigh_y) * np.hypot(shank_x, shank_y)
        )
    cosine = np.clip(
        np.nan_to_num(cosine, nan=0.0, posinf=1.0, neginf=-1.0),
        -1.0,
        1.0,
    )
    return np.arccos(cosine)


def _foot_lift(cleaned, valid, foot_joint):
    """Toe height above its own median, smoothed (positive while lifted)."""
    foot_y = _joint_xy(cleaned, valid, foot_joint)[1]
    if foot_y is None:
        return None
    baseline = float(np.nanmedian(foot_y))
    return _smooth(baseline - foot_y)


def _stance_swing_ratio(lift):
    if lift is None:
        return np.nan
    spread = _range95(lift)
    if not np.isfinite(spread) or spread < 1e-4:
        return np.nan
    swing = lift > 0.2 * spread
    stance = ~swing
    if int(swing.sum()) < 2:
        return np.nan
    return float(stance.sum()) / float(swing.sum())


def raw_gait_parameters(record):
    """Compute the raw-pose gait proxies for one cached sequence.

    REAL MODE: frame timing comes from the true fps and the raw frame count,
    and the continuous phase comes from the Hilbert transform of the centered
    left-ankle x signal, resampled onto the prepared FRAMES grid.

    SMOKE MODE (explicit branch): synthetic records carry no fps, so this
    function substitutes fps = 30.0 and duration =
    len(sequence) / fps. Because the fixtures were generated from a known
    linear clock of 0 to 4*pi radians over the clip, the smoke branch
    synthesizes the phase signal directly from that generator rule instead of
    trusting footfall detection: phase(t) = (4*pi * t / T) mod (2*pi), with
    exactly two stride cycles per fixture. The cadence target follows from
    those two cycles. Smoke output is a fixture, never a clinical result.
    """
    synthetic = MODE == "smoke"
    sequence = np.asarray(record["sequence"], dtype=np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
        raise ValueError(f"Bad raw pose shape: {sequence.shape}")
    total_frames = len(sequence)
    if synthetic:
        fps = 30.0
        frame_numbers = np.arange(total_frames)
    else:
        fps = float(record["fps"])
        with np.load(record["path"], allow_pickle=False) as data:
            frame_numbers = np.asarray(data["frame_numbers"])
        if len(frame_numbers) != total_frames:
            raise ValueError("frame_numbers do not match the pose count")
    duration_seconds = total_frames / fps

    cleaned, valid = interpolate_low_visibility(
        sequence, threshold=0.45, max_gap=4
    )
    pelvis_x, pelvis_y = _pelvis_xy(cleaned, valid)

    result = {
        "sequence_id": record["sequence_id"],
        "condition": record["condition"],
        "video_id": record["video_id"],
        "source_frames": total_frames,
        "fps": fps,
        "duration_seconds": duration_seconds,
        "cadence_cycles_per_second": np.nan,
        "step_length_asymmetry": np.nan,
        "knee_excursion_radians": np.nan,
        "trunk_sway_normalized_units": np.nan,
        "speed_proxy_normalized_per_second": np.nan,
        "stance_swing_ratio_proxy": np.nan,
        "phase_aligned": np.full(FRAMES, np.nan, dtype=np.float64),
    }

    # Trunk sway and speed proxy use the pelvis in raw image coordinates.
    if pelvis_x is not None:
        result["trunk_sway_normalized_units"] = float(
            np.nanstd(_detrend(pelvis_x))
        )
        dx = np.diff(_smooth(pelvis_x))
        dy = np.diff(_smooth(pelvis_y))
        result["speed_proxy_normalized_per_second"] = float(
            np.nanmean(np.hypot(dx, dy)) * fps
        )

    # Knee excursion is the range of a 2-D hip-knee-ankle angle proxy.
    excursions = [
        _range95(series)
        for series in (
            _knee_angle_series(cleaned, valid, 23, 25, 27),
            _knee_angle_series(cleaned, valid, 24, 26, 28),
        )
        if series is not None
    ]
    if excursions:
        result["knee_excursion_radians"] = float(np.mean(excursions))

    # Stance/swing ratio proxy from the toe lift above its median height.
    ratios = [
        _stance_swing_ratio(_foot_lift(cleaned, valid, joint))
        for joint in (31, 32)
    ]
    result["stance_swing_ratio_proxy"] = float(np.nanmean(ratios))

    # Step-length asymmetry from the horizontal ankle excursion range.
    left_x = _relative_ankle_x(cleaned, valid, 27, pelvis_x)
    right_x = _relative_ankle_x(cleaned, valid, 28, pelvis_x)
    if left_x is not None and right_x is not None:
        left_range = _range95(_smooth(left_x))
        right_range = _range95(_smooth(right_x))
        center = 0.5 * (left_range + right_range)
        if np.isfinite(center) and center > 1e-4:
            result["step_length_asymmetry"] = float(
                abs(right_range - left_range) / center
            )

    # Cadence and continuous phase share one left-ankle clock.
    ankle_fraction = float(np.isfinite(cleaned[:, 27, 0]).mean())
    clock_ok = pelvis_x is not None and ankle_fraction >= 0.4
    if synthetic:
        # SMOKE BRANCH (labelled): synthesized fps, duration and phase.
        unwrapped = 4.0 * np.pi * np.arange(total_frames) / total_frames
        if total_frames != FRAMES:
            unwrapped = temporal_resize(unwrapped, FRAMES)
        result["phase_aligned"] = np.mod(unwrapped, 2.0 * np.pi)
        result["cadence_cycles_per_second"] = 2.0 / duration_seconds
    elif clock_ok:
        clock = _relative_ankle_x(cleaned, valid, 27, pelvis_x)
        finite = np.isfinite(clock) if clock is not None else None
        if clock is not None and finite is not None and _range95(clock) > 1e-4:
            # Use the longest continuous observed run for the phase clock.
            # Frames outside that run stay NaN so they never enter probes,
            # matching the short-gap-only interpolation policy.
            boundaries = np.flatnonzero(
                np.diff(np.concatenate([[0], finite.astype(int), [0]]))
            )
            run_start = boundaries[0::2]
            run_end = boundaries[1::2]
            if len(run_start):
                best = int(np.argmax(run_end - run_start))
                clock_run = clock[run_start[best]:run_end[best]]
                unwrapped = np.full(total_frames, np.nan, dtype=np.float64)
                if len(clock_run) >= 8:
                    analytic = scipy.signal.hilbert(_smooth(clock_run))
                    run_phase = np.unwrap(np.angle(analytic))
                    unwrapped[run_start[best]:run_end[best]] = run_phase
                    low = int(0.05 * len(clock_run))
                    high = int(0.95 * len(clock_run))
                    if high - low >= 2:
                        cycles = (run_phase[high - 1] - run_phase[low]) / (
                            2.0 * np.pi
                        )
                        interior_seconds = (high - 1 - low) / fps
                        if interior_seconds > 0:
                            result["cadence_cycles_per_second"] = float(
                                cycles / interior_seconds
                            )
                if total_frames != FRAMES:
                    unwrapped = temporal_resize(unwrapped, FRAMES)
                result["phase_aligned"] = np.mod(unwrapped, 2.0 * np.pi)
    return result

In [ ]:
# Run the proxy extraction over every cached sequence and persist the table.
rows = []
aligned_phases = []
for record in records:
    parameters = raw_gait_parameters(record)
    aligned_phases.append(parameters.pop("phase_aligned"))
    rows.append(parameters)
gait_table = pd.DataFrame(rows)
gait_table.to_csv(ARTIFACT_DIR / "08_gait_parameters.csv", index=False)
phase_matrix = np.stack(aligned_phases)
display(
    gait_table.groupby("condition")[GAIT_PARAMETER_COLUMNS]
    .mean()
    .round(4)
)
missing_summary = gait_table[GAIT_PARAMETER_COLUMNS].isna().mean().round(3)
print("gait parameter table:", gait_table.shape)
print("missing fraction per parameter:")
print(missing_summary.to_string())
print(
    "phase matrix:", phase_matrix.shape,
    "(NaN frames mean the ankle clock failed for that sequence)",
)

## Pool one vector per sequence

The S-JEPA paper does not specify a downstream pooling, so this tutorial keeps the notebook 05 contract: concatenate four validity-masked statistics.

1. mean over every joint-time token
2. standard deviation over every joint-time token
3. mean over the ten neurologic joints
4. standard deviation over the ten neurologic joints

A patch contributes only when all four source frames are valid for that joint, so zero sentinels never enter the statistics directly. With the real checkpoint the token dimension is 96, which yields the 384-d vector promised by the notebook title. The code below reads the dimension from the checkpoint config, so smoke checkpoints (dimension 32) work unchanged.

In [ ]:
def masked_mean_std(tokens, mask):
    weights = torch.as_tensor(
        mask, dtype=tokens.dtype, device=tokens.device
    ).unsqueeze(-1)
    denominator = weights.sum(dim=1).clamp_min(1.0)
    mean = (tokens * weights).sum(dim=1) / denominator
    variance = (
        (tokens - mean[:, None, :]).square() * weights
    ).sum(dim=1) / denominator
    return mean, variance.clamp_min(0.0).sqrt()


@torch.no_grad()
def pooled_embeddings(model, arrays, validity, batch_size=8):
    vectors = []
    segments = model.target_encoder.segments
    segment_length = model.target_encoder.segment_length
    dimension = model.target_encoder.embed_dim
    for start in range(0, len(arrays), batch_size):
        batch = torch.tensor(
            arrays[start:start + batch_size],
            dtype=torch.float32,
        )
        tokens = model.target_encoder(batch).reshape(
            len(batch), segments, 33, dimension
        )
        valid_patch = np.asarray(
            validity[start:start + batch_size], dtype=bool
        ).reshape(
            len(batch), segments, segment_length, 33
        ).all(axis=2)
        global_tokens = tokens.reshape(len(batch), -1, dimension)
        neuro_tokens = tokens[:, :, MASK_KEYPOINTS].reshape(
            len(batch), -1, dimension
        )
        global_mean, global_std = masked_mean_std(
            global_tokens, valid_patch.reshape(len(batch), -1)
        )
        neuro_mean, neuro_std = masked_mean_std(
            neuro_tokens,
            valid_patch[:, :, MASK_KEYPOINTS].reshape(len(batch), -1),
        )
        vector = torch.cat(
            [
                global_mean,
                global_std,
                neuro_mean,
                neuro_std,
            ],
            dim=1,
        )
        vectors.append(vector.cpu())
    return torch.cat(vectors).numpy()


embeddings = pooled_embeddings(model, all_xyz, all_valid)
print("embedding matrix:", embeddings.shape)
assert np.isfinite(embeddings).all()

## Probe protocol: three feature families, leave-one-out

Each scalar gait parameter is probed three times with the identical fitting procedure, so the only difference between rows is the feature family:

- sjepa_pooled_latents, the frozen 4-block pool described above;
- raw_coordinate_pool_stats, the same masked mean/std recipe applied directly to the prepared raw coordinates (12 features: mean and std over each coordinate channel, global and neurologic);
- missingness_features, the per-joint and per-frame detector validity fractions used in notebook 06.

Every probe standardizes features inside the training fold, picks the ridge alpha with RidgeCV, and scores leave-one-sequence-out predictions with R2. Sequences that share a source video can still leak through this split, which is exactly why the baselines matter and why notebook 07 audits source identity separately.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, LeaveOneOut


def _standardize(train, test):
    mean = train.mean(axis=0)
    std = train.std(axis=0)
    std = np.where(std < 1e-12, 1.0, std)
    return (train - mean) / std, (test - mean) / std


def ridge_loo_r2(features, target, alphas=None):
    """Leave-one-sequence-out out-of-fold R2 for one scalar target."""
    if alphas is None:
        alphas = np.logspace(-3.0, 3.0, 13)
    features = np.asarray(features, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    keep = np.isfinite(target) & np.isfinite(features).all(axis=1)
    n_keep = int(keep.sum())
    if n_keep < 6:
        return np.nan, n_keep, "too few finite rows"
    x, y = features[keep], target[keep]
    if float(np.std(y)) < 1e-12:
        return np.nan, n_keep, "target has no variance"
    y_true, y_pred = [], []
    for train_index, test_index in LeaveOneOut().split(x):
        x_train, x_test = _standardize(x[train_index], x[test_index])
        probe = RidgeCV(alphas=alphas).fit(x_train, y[train_index])
        y_pred.append(float(probe.predict(x_test)[0]))
        y_true.append(float(y[test_index][0]))
    return float(r2_score(y_true, y_pred)), n_keep, ""


# Baseline family (a): identical masked mean/std pooling of raw coordinates.
def raw_coordinate_pool_stats(arrays, validity):
    rows = []
    for sample in range(len(arrays)):
        features = []
        for channel in range(3):
            values = arrays[sample, ..., channel]
            mask = validity[sample]
            neuro_mask = mask[:, MASK_KEYPOINTS]
            blocks = (values[mask], values[:, MASK_KEYPOINTS][neuro_mask])
            for block in blocks:
                features.append(float(np.mean(block)))
                features.append(float(np.std(block)))
        rows.append(features)
    return np.asarray(rows, dtype=np.float32)


# Baseline family (b): detector missingness features (same as notebook 06).
raw_stats = raw_coordinate_pool_stats(all_xyz, all_valid)
missingness_features = np.concatenate(
    [
        all_valid.mean(axis=1),
        all_valid.mean(axis=2),
    ],
    axis=1,
).astype(np.float32)

feature_families = {
    "sjepa_pooled_latents": embeddings,
    "raw_coordinate_pool_stats": raw_stats,
    "missingness_features": missingness_features,
}
for name, features in feature_families.items():
    print(f"feature family '{name}': {features.shape}")

probe_rows = []
for parameter in GAIT_PARAMETER_COLUMNS:
    target = gait_table[parameter].to_numpy(dtype=np.float64)
    for family, features in feature_families.items():
        r2, n_keep, note = ridge_loo_r2(features, target)
        probe_rows.append({
            "parameter": parameter,
            "feature_family": family,
            "n_sequences": n_keep,
            "out_of_fold_r2": r2,
            "note": note,
        })
probe_scores = pd.DataFrame(probe_rows)
probe_scores.to_csv(
    ARTIFACT_DIR / "08_gait_parameter_probe_scores.csv", index=False
)
display(
    probe_scores.pivot(
        index="parameter", columns="feature_family", values="out_of_fold_r2"
    )
    .reindex(GAIT_PARAMETER_COLUMNS)
    .round(3)
)
print("probe score rows:", len(probe_scores))

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(11, 4.5))
width = 0.26
positions = np.arange(len(GAIT_PARAMETER_COLUMNS))
colors = {"sjepa_pooled_latents": "#17324d",
          "raw_coordinate_pool_stats": "#ef7d57",
          "missingness_features": "#b9d6c6"}
for offset, family in enumerate(feature_families):
    values = np.asarray([
        probe_scores.loc[
            (probe_scores["parameter"] == parameter)
            & (probe_scores["feature_family"] == family),
            "out_of_fold_r2",
        ].iloc[0]
        for parameter in GAIT_PARAMETER_COLUMNS
    ], dtype=np.float64)
    # A skipped probe (for example a target without variance) is drawn flat
    # so the axis stays readable; the score table keeps the NaN.
    values = np.nan_to_num(values, nan=0.0)
    axis.bar(
        positions + (offset - 1.0) * width,
        values,
        width,
        label=family,
        color=colors[family],
    )
axis.set_xticks(positions)
axis.set_xticklabels(GAIT_PARAMETER_COLUMNS, rotation=16, ha="right")
axis.axhline(0.0, color="black", linewidth=0.8)
axis.set_ylabel("leave-one-sequence-out R2")
axis.set_ylim(-0.8, 1.05)
axis.set_title(f"Gait parameter probes ({MODE} mode)")
axis.legend(loc="lower right", fontsize=8)
figure.tight_layout()
figure.savefig(
    ARTIFACT_DIR / "08_gait_parameter_probe_scores.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()
print("saved: 08_gait_parameter_probe_scores.png")

## Probe the phase per frame: the latent phase clock

Cadence and symmetry are sequence-level properties. Gait phase is different: it advances inside every clip, so the sharpest test of the representation is per frame. The target encoder produced one token per joint-time segment (16 segments of 4 frames each). We mean the joint tokens inside each segment into a single latent vector of dimension D, broadcast that vector to the 4 frames of the segment, and try to read the continuous phase at each frame with one Ridge map to (cos, sin).

This is deliberately a segment-granularity clock: all 4 frames inside a segment share one latent, so phase differences WITHIN a segment cannot be resolved. What the probe can measure is whether the latent knows where each segment sits in the stride cycle. Evaluation uses grouped 5-fold CV with whole sequences as groups, because neighbouring frames are heavily correlated and a random split would overstate R2.

If the probe works, the predicted angle tracks the true angle around the circle and the polar spokes are short. If the pretraining never produced phase structure, predictions collapse toward a constant and the plot becomes a shapeless cloud. The smoke fixtures use the synthesized phase signal, so a smoke result proves the code path only.

In [ ]:
def segment_latent_features(tokens, patch_valid):
    """Mean the joint tokens of each segment into one latent vector.

    tokens: (N, segments, joints, dim), patch_valid: (N, segments, joints).
    Invalid patches never contribute; an all-invalid segment falls back to
    the sequence-level mean of its valid tokens.
    """
    mask = np.asarray(patch_valid, dtype=bool)
    weights = mask.astype(np.float64)
    numerator = np.sum(tokens * weights[..., None], axis=2)
    denominator = weights.sum(axis=2)
    segment_mean = numerator / np.maximum(denominator, 1e-12)[..., None]
    segment_mean[denominator == 0] = np.nan
    total_num = np.sum(tokens * weights[..., None], axis=(1, 2))
    total_den = weights.sum(axis=(1, 2))
    sequence_mean = total_num / np.maximum(total_den, 1e-12)[:, None]
    broken = ~np.isfinite(segment_mean).all(axis=2)
    segment_mean = np.where(
        broken[..., None], sequence_mean[:, None, :], segment_mean
    )
    return segment_mean.astype(np.float32)


@torch.no_grad()
def frame_latents(model, arrays, validity, batch_size=8):
    """One latent per frame: the segment latent, broadcast to its 4 frames."""
    vectors = []
    segments = model.target_encoder.segments
    segment_length = model.target_encoder.segment_length
    dimension = model.target_encoder.embed_dim
    for start in range(0, len(arrays), batch_size):
        batch = torch.tensor(
            arrays[start:start + batch_size], dtype=torch.float32
        )
        tokens = model.target_encoder(batch).reshape(
            len(batch), segments, 33, dimension
        )
        patch_valid = np.asarray(
            validity[start:start + batch_size], dtype=bool
        ).reshape(len(batch), segments, segment_length, 33).all(axis=2)
        per_segment = segment_latent_features(
            tokens.cpu().numpy(), patch_valid
        )
        vectors.append(np.repeat(per_segment, segment_length, axis=1))
    return np.concatenate(vectors, axis=0).astype(np.float32)


def grouped_ridge_oof(features, targets, groups, n_splits=5, alphas=None):
    """Out-of-fold Ridge predictions with whole sequences as groups.

    Returns predictions in the original row order.
    """
    if alphas is None:
        alphas = np.logspace(-3.0, 3.0, 13)
    n_splits = min(n_splits, len(np.unique(groups)))
    oof = np.full_like(targets, np.nan, dtype=np.float64)
    for train_index, test_index in GroupKFold(
        n_splits=n_splits
    ).split(features, targets, groups):
        x_train, x_test = _standardize(
            features[train_index], features[test_index]
        )
        probe = RidgeCV(alphas=alphas).fit(x_train, targets[train_index])
        oof[test_index] = probe.predict(x_test)
    return oof


frame_latent_features = frame_latents(model, all_xyz, all_valid)
flat_phase = phase_matrix.reshape(-1)
keep_frame = np.isfinite(flat_phase)
row_index = np.flatnonzero(keep_frame)
sample_sequence = row_index // FRAMES
sample_frame = row_index % FRAMES
targets_phase = np.stack(
    [np.cos(flat_phase[keep_frame]), np.sin(flat_phase[keep_frame])], axis=1
)
features_phase = frame_latent_features.reshape(
    -1, frame_latent_features.shape[-1]
)[keep_frame]
print(
    "phase probe rows:", int(keep_frame.sum()),
    "token dim:", frame_latent_features.shape[-1],
)

if int(keep_frame.sum()) >= 40:
    oof_phase = grouped_ridge_oof(
        features_phase, targets_phase, sample_sequence, n_splits=5
    )
    r2_cos = float(r2_score(targets_phase[:, 0], oof_phase[:, 0]))
    r2_sin = float(r2_score(targets_phase[:, 1], oof_phase[:, 1]))
    r2_vector = 0.5 * r2_cos + 0.5 * r2_sin
    predicted_angle = np.arctan2(oof_phase[:, 1], oof_phase[:, 0])
    angular_error = np.mod(
        predicted_angle - flat_phase[keep_frame] + np.pi, 2.0 * np.pi
    ) - np.pi
    phase_metrics = pd.DataFrame({
        "metric": ["r2_cos", "r2_sin", "r2_vector_mean"],
        "value": [r2_cos, r2_sin, r2_vector],
    })
    phase_metrics.to_csv(
        ARTIFACT_DIR / "08_phase_probe_metrics.csv", index=False
    )
    display(phase_metrics.round(4))
    prediction_table = pd.DataFrame({
        "sequence_id": sequence_ids[sample_sequence],
        "condition": labels[sample_sequence],
        "video_id": video_ids[sample_sequence],
        "frame": sample_frame,
        "phase_true_radians": flat_phase[keep_frame],
        "phase_pred_radians": predicted_angle,
        "cos_true": targets_phase[:, 0],
        "sin_true": targets_phase[:, 1],
        "cos_pred": oof_phase[:, 0],
        "sin_pred": oof_phase[:, 1],
        "angular_error_degrees": np.degrees(np.abs(angular_error)),
    })
    prediction_table.to_csv(
        ARTIFACT_DIR / "08_phase_probe_predictions.csv", index=False
    )
    print("saved: 08_phase_probe_metrics.csv, 08_phase_probe_predictions.csv")
else:
    oof_phase = None
    print("Too few frames with a usable phase signal to run the grouped CV.")

# Supplementary probe: left-ankle tokens only. The phase clock is defined by
# the LEFT ankle, while the probe above averages all 33 joints per segment.
# Left and right ankles oscillate in anti-phase, so joint averaging can cancel
# the very signal the probe looks for. This check repeats the grouped ridge
# probe using only joint 27 segment latents against the left-ankle phase.
@torch.no_grad()
def joint_segment_latents(model, arrays, validity, joints, batch_size=8):
    vectors = []
    segments = model.target_encoder.segments
    segment_length = model.target_encoder.segment_length
    dimension = model.target_encoder.embed_dim
    for start in range(0, len(arrays), batch_size):
        batch = torch.tensor(
            arrays[start:start + batch_size], dtype=torch.float32
        )
        tokens = model.target_encoder(batch).reshape(
            len(batch), segments, 33, dimension
        )
        patch_valid = np.asarray(
            validity[start:start + batch_size], dtype=bool
        ).reshape(len(batch), segments, segment_length, 33).all(axis=2)
        selected = tokens[:, :, joints, :].cpu().numpy()
        selected_valid = patch_valid[:, :, joints]
        weights = selected_valid.astype(np.float64)[..., None]
        denominator = weights.sum(axis=2)
        mean = (selected * weights).sum(axis=2) / np.maximum(
            denominator, 1e-12
        )
        mean[denominator[..., 0] == 0] = np.nan
        vectors.append(np.repeat(mean, segment_length, axis=1))
    return np.concatenate(vectors, axis=0).astype(np.float32)


ankle_features = joint_segment_latents(
    model, all_xyz, all_valid, joints=[27]
)
ankle_features_flat = ankle_features.reshape(
    -1, ankle_features.shape[-1]
)[keep_frame]
ankle_keep = np.isfinite(ankle_features_flat).all(axis=1)
ankle_rows = int(ankle_keep.sum())
print("left-ankle probe rows:", ankle_rows)

if ankle_rows >= 40:
    oof_ankle = grouped_ridge_oof(
        ankle_features_flat[ankle_keep],
        targets_phase[ankle_keep],
        sample_sequence[ankle_keep],
        n_splits=5,
    )
    r2_cos_ankle = float(
        r2_score(targets_phase[ankle_keep, 0], oof_ankle[:, 0])
    )
    r2_sin_ankle = float(
        r2_score(targets_phase[ankle_keep, 1], oof_ankle[:, 1])
    )
    ankle_metrics = pd.DataFrame({
        "metric": ["r2_cos", "r2_sin", "r2_vector_mean"],
        "value": [
            r2_cos_ankle, r2_sin_ankle,
            0.5 * r2_cos_ankle + 0.5 * r2_sin_ankle,
        ],
    })
    ankle_metrics.to_csv(
        ARTIFACT_DIR / "08_phase_probe_ankle_metrics.csv", index=False
    )
    display(ankle_metrics.round(4))
    print("saved: 08_phase_probe_ankle_metrics.csv")
else:
    print("Too few frames for the left-ankle probe.")


In [ ]:
import matplotlib.pyplot as plt

if oof_phase is not None:
    chosen_sequences = []
    for sequence_index in range(len(phase_matrix)):
        if len(chosen_sequences) >= 3:
            break
        in_sequence = sample_sequence == sequence_index
        if int(in_sequence.sum()) >= 8:
            chosen_sequences.append(sequence_index)
    if not chosen_sequences:
        chosen_sequences = list(range(min(3, len(phase_matrix))))

    figure, axes = plt.subplots(
        1,
        len(chosen_sequences),
        figsize=(4.0 * len(chosen_sequences), 3.6),
        subplot_kw={"projection": "polar"},
        squeeze=False,
    )
    for axis, sequence_index in zip(axes[0], chosen_sequences):
        in_sequence = np.flatnonzero(sample_sequence == sequence_index)
        subsample = in_sequence[::2]
        true_angle = flat_phase[row_index[subsample]]
        pred_angle = predicted_angle[subsample]
        for true_value, pred_value in zip(true_angle, pred_angle):
            axis.plot(
                [true_value, pred_value],
                [0.95, 0.55],
                color="#c8c8c8",
                linewidth=0.7,
                zorder=1,
            )
        axis.scatter(
            true_angle, np.full(len(true_angle), 0.95),
            s=16, color="0.35", label="true phase", zorder=2,
        )
        axis.scatter(
            pred_angle, np.full(len(pred_angle), 0.55),
            s=14, marker="x", color="#17324d",
            label="predicted phase", zorder=3,
        )
        sequence_errors = np.mod(
            predicted_angle[in_sequence]
            - flat_phase[row_index[in_sequence]]
            + np.pi,
            2.0 * np.pi,
        ) - np.pi
        mean_error = float(np.degrees(np.mean(np.abs(sequence_errors))))
        axis.set_title(
            f"{labels[sequence_index]} {sequence_ids[sequence_index]}\n"
            f"mean abs circular error {mean_error:.1f} deg",
            fontsize=8,
        )
        axis.legend(loc="lower left", fontsize=7, frameon=False)
    figure.suptitle(
        f"Per-frame phase probe: predicted against true phase ({MODE} mode)",
        fontsize=11,
    )
    figure.tight_layout()
    figure.savefig(
        ARTIFACT_DIR / "08_phase_probe_polar.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    print("saved: 08_phase_probe_polar.png")

## Interpretation: what the real run actually found

Read this section with the measured tables, not the hypotheses. The real-mode run produced mostly nulls, and the paper should report them as such.

1. Which parameters are linearly decodable from the frozen pool? Only knee excursion clears a meaningful bar (out-of-fold R2 near 0.30), and it does so equally in the raw-coordinate and missingness families, so pretraining added nothing measurable there. Cadence (R2 about 0.01), sway (0.04), speed (0.06), asymmetry (0.09), and the stance/swing proxy (about 0) are weak or null. Absolute cadence is additionally fragile as a target: it is computed from a single-camera ankle clock, some clips contain fewer than two estimated cycles, and the horizontal ankle channel is structurally blind when a person walks toward or away from the camera. The honest sentence is that the frozen pool does not linearly retain most canonical gait parameters, not that the parameters were unlearnable in principle.

2. Does pretraining help over the raw-coordinate baseline? Only marginally, and only for speed and sway, where the latent family beats a weak raw baseline; for knee excursion all families tie. The pooled families also differ in dimensionality (384 versus 12 versus 97), so the comparison is informative, not complexity-matched.

3. Is there a latent phase clock? The per-frame probe (all 33 joints averaged per segment) scores negative R2 on cos and sin, and the supplementary left-ankle-only probe reports its own numbers in the output above. Gait is bilaterally anti-phase, so joint averaging can cancel the stride-frequency signal; only after reading both probes may the paper claim whether phase structure survives in token latents. If both are null, say so plainly and treat the absence of a linear phase clock as a measured fact about this pooling, not about all possible readouts.

Limits to state in the paper: every gait quantity is a 2-D video proxy from uncalibrated normalized coordinates; the corpus is small and several sequences share a source upload, so leave-one-sequence-out can still leak video identity (treat these R2 values as upper bounds); the normal sequences also served as pretraining data, which is transductive; and smoke-mode tables validate code paths only. A probe is a measurement tool for the workshop question, not a diagnosis.

## Before you write it up

- Read the probe table together with notebook 06 (classifier lanes) and notebook 07 (source-identity audit). A probe that tracks source videos is not a gait result.
- Report the baselines beside the latent scores, never the latent scores alone.
- State which mode produced each number, and that all gait quantities are video proxies.
- In smoke mode the cadence target is constant by construction and the phase is synthesized, so expect skipped probes there. That is the fixture speaking, not the model.